# ISIC experiments on Colab from GitHub
Run cells in order on a Colab GPU. Code is pinned to a tested commit in your fork. MODE defaults to a one-epoch smoke test; use train or evaluate for full experiments; MODELS selects which models run. New runs receive unique output directories. For recovery, set RESUME_RUN to a prior run created by the updated code and select models with last.pt. Legacy best.pt files cannot provide an exact resume.

Progress includes phase/batch ETA. Each completed epoch saves last.pt before reporting completion. A disconnect during an epoch loses that unfinished epoch. CPU zero-worker resume is regression-tested for identical weights; persistent worker augmentation states and CUDA numerical determinism are not guaranteed across restarts. Google Drive writes still depend on the remote filesystem; the notebook cannot prevent Colab runtime loss.


In [1]:
import sys, platform, subprocess, json, hashlib, shutil, time
from pathlib import Path
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
assert sys.platform == "linux" and Path("/content").is_dir(), "Select the remote Colab kernel, not local Python."
assert torch.cuda.is_available(), "Select a Colab GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
subprocess.run(["nvidia-smi"], check=True)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch: 2.14.1+cu130 CUDA build: 13.0
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


CompletedProcess(args=['nvidia-smi'], returncode=0)

## GitHub source and user-provided data
This public repository includes code, frozen split CSVs, duplicate-pair metadata and report figures. It does **not** include the full ISIC JPEG dataset, archive or trained weights, and does not grant access to the author's Google Drive.

Obtain your own SIIM-ISIC 2020 Training JPEGs from the dataset provider under its access and usage terms. Upload/extract them on the Colab runtime and set IMAGE_DIR, or set ARCHIVE to your uploaded ZIP for automatic extraction. Images must match the IDs in the included manifests. Google Drive is optional and uses your signed-in account. Configure OUTPUT_ROOT outside the repository; /content is temporary. Evaluation requires your own checkpoint paths. The installed Colab environment is recorded rather than silently replaced.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")
print("Drive mounted:", drive.is_dir())

archive = drive / "COMP3710/datasets/ISIC2020/ISIC_2020_Training_JPEG.zip"
print("Dataset ZIP accessible:", archive.is_file())

Drive mounted: True
Dataset ZIP accessible: True


In [5]:
REPO_URL = "https://github.com/Nandos-tm/PatternAnalysis-2026.git"
COMMIT = "5ce0a02aff506504a212b11e2de8e6d71ef2110d"
REPO = Path("/content/PatternAnalysis-2026-" + COMMIT[:7])
# Each user supplies their own ISIC 2020 JPEGs; GitHub contains no image archive.
# Paths refer to the REMOTE Colab filesystem, not your local computer.
USE_GOOGLE_DRIVE = True  # optional: mounts the signed-in user's own Drive
IMAGE_DIR = Path("/content/isic2020/jpeg")  # existing JPEG tree or extraction destination
ARCHIVE = Path("/content/drive/MyDrive/COMP3710/datasets/ISIC2020/ISIC_2020_Training_JPEG.zip")  # e.g. Path("/content/ISIC_2020_Training_JPEG.zip")
OUTPUT_ROOT = Path("/content/drive/MyDrive/COMP3710/isic_runs")  # temporary: download results before runtime ends
# For persistent results, enable Drive and choose your own folder, e.g.
# OUTPUT_ROOT = Path("/content/drive/MyDrive/isic_runs")
MODE = "train"  # "smoke": one epoch each; "train": full run; "evaluate": saved checkpoints
CHECKPOINTS = {"baseline": None, "siamese_aux": None}  # set Path(...) for evaluation
RESUME_RUN = None  # Path to an existing run under your own OUTPUT_ROOT
MODELS = ("baseline", "siamese_aux")
LOG_EVERY = 25
WORKERS = 2
BATCH_SIZE = 64  # evaluation only; training retains each script's defaults
RUN_TESTS = True
assert MODE in ("smoke", "evaluate", "train")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == COMMIT, f"Unexpected checkout: {actual}"
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip(), "Checkout has local changes."
PROJECT = REPO / "recognition/Siamese_Nandos"
SPLITS = PROJECT / "splits"
print("Using:", REPO, actual)


Using: /content/PatternAnalysis-2026-5ce0a02 5ce0a02aff506504a212b11e2de8e6d71ef2110d


In [6]:
# Dependency repair used in the prior Colab session.
%pip install jedi==0.20.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.6 MB/s  0:00:00


In [7]:
# Inspect package compatibility before running; preserve the current CUDA build.
from importlib.metadata import version
packages = ["torch", "torchvision", "numpy", "pandas", "Pillow", "scikit-learn", "matplotlib"]
versions = {name: version(name) for name in packages}
print(json.dumps(versions, indent=2))
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print(pip_check.stdout or pip_check.stderr or "No dependency conflicts found.")

if pip_check.returncode:
    raise RuntimeError("pip check found dependency conflicts; see output above.")
import pandas as pd
sys.path.insert(0, str(PROJECT))
from split_utils import load_splits
frames = load_splits(SPLITS)
audit = json.loads((SPLITS / "manifest_audit.json").read_text())
for name, details in audit["splits"].items():
    assert hashlib.sha256((SPLITS / f"{name}.csv").read_bytes()).hexdigest() == details["sha256"]
assert hashlib.sha256((PROJECT / "ISIC_2020_Training_Duplicates.csv").read_bytes()).hexdigest() == audit["duplicates"]["sha256"]
print("All manifest hashes and leakage checks passed.")
for name, frame in frames.items():
    print(name, len(frame), "images;", int(frame.target.sum()), "melanomas")
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT), "-v"], cwd=REPO, check=True)


{
  "torch": "2.14.1+cu130",
  "torchvision": "0.29.1+cu130",
  "numpy": "2.4.6",
  "pandas": "2.2.3",
  "Pillow": "11.3.0",
  "scikit-learn": "1.6.1",
  "matplotlib": "3.10.0"
}
No broken requirements found.

All manifest hashes and leakage checks passed.
train 23188 images; 403 melanomas
val 4938 images; 92 melanomas
test 5000 images; 89 melanomas


## Optional storage connection and output setup
Drive is mounted only when USE_GOOGLE_DRIVE is True. Complete authorization yourself if prompted. Otherwise use files uploaded to the runtime. Store outputs outside the repository; download local /content results before the session ends, or choose a persistent folder in your own Drive. No dataset or checkpoint download is performed automatically.


In [8]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
IMAGE_DIR = Path(IMAGE_DIR)
OUTPUT_ROOT = Path(OUTPUT_ROOT)
ARCHIVE = Path(ARCHIVE) if ARCHIVE is not None else None
for location in (IMAGE_DIR, OUTPUT_ROOT, ARCHIVE, RESUME_RUN):
    if location is not None and Path(location).resolve().is_relative_to(REPO.resolve()):
        raise ValueError("Keep datasets, archives and run outputs outside the Git checkout.")
if not IMAGE_DIR.is_dir() and ARCHIVE is None:
    raise FileNotFoundError("Supply your own JPEG directory as IMAGE_DIR, or set ARCHIVE to your ISIC 2020 ZIP.")
if MODE == "evaluate":
    for name in MODELS:
        checkpoint = CHECKPOINTS.get(name)
        if checkpoint is None or not Path(checkpoint).is_file():
            raise FileNotFoundError(f"Set CHECKPOINTS[{name!r}] to your own existing checkpoint path.")
        CHECKPOINTS[name] = Path(checkpoint)
if not OUTPUT_ROOT.resolve().is_relative_to(Path("/content/drive")):
    print("Outputs use runtime storage. Download them before Colab disconnects.")
from uuid import uuid4
RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid4().hex[:8]
if RESUME_RUN is not None:
    assert MODE == "train", "Resume applies only to training."
    OUTPUT = Path(RESUME_RUN)
    assert OUTPUT.is_dir(), f"Resume directory missing: {OUTPUT}"
    for model_name in MODELS:
        assert (OUTPUT / "training" / model_name / "last.pt").is_file(), (
            f"No resumable checkpoint for {model_name}. Select only models with last.pt in MODELS; "
            "best.pt from the old code cannot resume exactly.")
else:
    OUTPUT = OUTPUT_ROOT / RUN_ID
    OUTPUT.mkdir(parents=True, exist_ok=False)
metadata = {"commit": COMMIT, "mode": MODE, "packages": versions,
            "python": sys.version, "gpu": torch.cuda.get_device_name(0),
            "cuda_build": torch.version.cuda, "manifest_audit": audit,
            "inference_precision": "FP16 CUDA autocast (predict.py)",
            "workers": WORKERS, "evaluation_batch_size": BATCH_SIZE,
            "models": list(MODELS), "resume": RESUME_RUN is not None}
(OUTPUT / f"environment_{RUN_ID}.json").write_text(json.dumps(metadata, indent=2))
print("Results will be saved to:", OUTPUT)


Results will be saved to: /content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4


## Extract required JPEGs to local Colab storage
Evaluation needs validation/test images. Training also needs training images. Extraction writes only normalized JPEG filenames and checks archive completeness, duplicate IDs and available space.


In [9]:
from zipfile import ZipFile
from dataset import index_images
parts = ("train", "val", "test") if MODE != "evaluate" else ("val", "test")
needed = {name for part in parts for name in frames[part].image_name}
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
paths = index_images(IMAGE_DIR)
missing = needed - paths.keys()
if missing:
    assert ARCHIVE is not None and ARCHIVE.is_file(), (
        f"{len(missing)} JPEGs missing from IMAGE_DIR. Supply a complete image copy or set ARCHIVE to your own ZIP.")
    with ZipFile(ARCHIVE) as archive:
        members = {}
        for member in archive.infolist():
            key = Path(member.filename).stem
            if Path(member.filename).suffix.lower() in (".jpg", ".jpeg") and key in missing:
                if key in members:
                    raise ValueError(f"Duplicate archive image ID: {key}")
                members[key] = member
        assert missing <= members.keys(), f"Archive lacks {len(missing - members.keys())} required images"
        required_bytes = sum(m.file_size for m in members.values())
        assert shutil.disk_usage(IMAGE_DIR).free > required_bytes + 2 * 1024**3, "Insufficient local disk space"
        for i, (key, member) in enumerate(members.items(), 1):
            target = IMAGE_DIR / (key + Path(member.filename).suffix.lower())
            temporary = target.with_suffix(".partial")
            with archive.open(member) as source, temporary.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            assert temporary.stat().st_size == member.file_size
            temporary.replace(target)
            if i % 1000 == 0:
                print(f"Extracted {i}/{len(members)}", flush=True)
paths = index_images(IMAGE_DIR)
assert needed <= paths.keys()
print("Required JPEGs available:", len(needed))


Extracted 1000/33126
Extracted 2000/33126
Extracted 3000/33126
Extracted 4000/33126
Extracted 5000/33126
Extracted 6000/33126
Extracted 7000/33126
Extracted 8000/33126
Extracted 9000/33126
Extracted 10000/33126
Extracted 11000/33126
Extracted 12000/33126
Extracted 13000/33126
Extracted 14000/33126
Extracted 15000/33126
Extracted 16000/33126
Extracted 17000/33126
Extracted 18000/33126
Extracted 19000/33126
Extracted 20000/33126
Extracted 21000/33126
Extracted 22000/33126
Extracted 23000/33126
Extracted 24000/33126
Extracted 25000/33126
Extracted 26000/33126
Extracted 27000/33126
Extracted 28000/33126
Extracted 29000/33126
Extracted 30000/33126
Extracted 31000/33126
Extracted 32000/33126
Extracted 33000/33126
Required JPEGs available: 33126


## Launch and log the experiment
Logs stream into the notebook and are saved to OUTPUT_ROOT. With `MODE='train'`, both models are trained from ImageNet initialization in new folders before held-out evaluation. With `MODE='evaluate'`, the original checkpoints are used. The validation threshold policy is fixed at 80% specificity.


In [10]:
import os, signal

# Repeat checks immediately before launching, even if setup cells ran out of order.
def preflight(parts):
    checked = load_splits(SPLITS)
    indexed = index_images(IMAGE_DIR)
    required = {image_id for part in parts for image_id in checked[part].image_name}
    missing = required - indexed.keys()
    if missing:
        raise FileNotFoundError(f"{len(missing)} required JPEGs are missing. Finish the extraction cell first. Examples: {sorted(missing)[:5]}")
    assert torch.cuda.is_available(), "CUDA is unavailable."
    print(f"Launch preflight passed: {len(required)} images and leakage audit.", flush=True)


def run_script(script, arguments, log_name):
    preflight(("train", "val") if script == "train.py" else ("val", "test"))
    command = [sys.executable, "-u", str(PROJECT / script), *map(str, arguments)]
    log_path = OUTPUT / f"{Path(log_name).stem}_{time.strftime('%H%M%S', time.gmtime())}_{uuid4().hex[:8]}.log"
    print("Running:", command, "\nLog:", log_path, flush=True)
    # Append-only attempt history; each attempt gets its own log.
    with (OUTPUT / "commands.jsonl").open("a") as record:
        record.write(json.dumps({"command": command, "log": str(log_path)}) + "\n")
    with log_path.open("x") as log:
        log.write(json.dumps(command) + "\n")
        log.flush()
        process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                   start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            # Colab is Linux: terminate the entire group, including DataLoader workers.
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
            # Kill any descendants that survived after the parent exited.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
            raise
        finally:
            process.stdout.close()
        if code:
            raise subprocess.CalledProcessError(code, command)



## One-epoch GPU smoke test

With the default `MODE="smoke"`, run the setup, extraction and helper cells above, then this cell. It trains both models from ImageNet initialization for one complete epoch using the frozen splits and normal batch/pair defaults, evaluates both checkpoints, and also evaluates Siamese prototypes. This is a real-data run, so it can take substantial time. All training, validation and test JPEGs are required. Outputs use new subfolders; logs and commands remain in `OUTPUT`. Later full-experiment cells are skipped in smoke mode.


In [16]:
# Run setup and extraction cells above first. MODE="smoke" prevents full training below.
if MODE == "smoke":
    assert RESUME_RUN is None, "Smoke tests use fresh output folders, not resume checkpoints."
    preflight(("train", "val", "test"))
    SMOKE_OUTPUT = OUTPUT / ("smoke_" + uuid4().hex[:8])
    SMOKE_OUTPUT.mkdir(exist_ok=False)
    smoke_rows = []
    for model_name in ("baseline", "siamese_aux"):
        train_output = SMOKE_OUTPUT / model_name / "training"
        evaluation = SMOKE_OUTPUT / model_name / "evaluation"
        common = ["--model", model_name, "--splits", SPLITS, "--images", IMAGE_DIR,
                  "--workers", WORKERS, "--device", "cuda"]
        run_script("train.py", [*common, "--output", train_output, "--epochs", 1,
                               "--log-every", LOG_EVERY], f"smoke_{model_name}_train.log")
        run_script("predict.py", [*common, "--checkpoint", train_output / "best.pt",
                                 "--output", evaluation, "--batch-size", BATCH_SIZE,
                                 "--threshold-policy", "val_specificity80_max_sensitivity",
                                 *(["--prototype"] if model_name == "siamese_aux" else [])],
                   f"smoke_{model_name}_evaluation.log")
        required_training = ["best.pt", "last.pt", "history.csv", "learning_curves.png", "run_manifest.json"]
        required_evaluation = ["test_metrics.csv", "test_predictions.csv", "validation_predictions.csv",
                               "resource_profile.csv", "calibration.csv", "confidence_summary.csv",
                               "failure_cases.csv", "run_manifest.json", "confusion_matrix.png",
                               "precision_recall_curve.png", "roc_curve.png", "reliability.png",
                               "confidence_correctness.png"]
        for folder, names in ((train_output, required_training), (evaluation, required_evaluation)):
            for name in names:
                artifact = folder / name
                assert artifact.is_file() and artifact.stat().st_size > 0, f"Missing/empty artifact: {artifact}"
        history = pd.read_csv(train_output / "history.csv")
        assert len(history) == 1 and int(history.iloc[0].epoch) == 1
        assert len(pd.read_csv(evaluation / "test_predictions.csv")) == len(frames["test"])
        metrics = pd.read_csv(evaluation / "test_metrics.csv")
        assert metrics[["test_accuracy", "test_macro_f1", "test_auroc", "test_pr_auc"]].notna().all().all()
        if model_name == "siamese_aux":
            for name in ("training_prototypes.npy", "test_metrics.csv", "val_predictions.csv", "test_predictions.csv"):
                assert (evaluation / "prototype" / name).is_file(), f"Missing prototype artifact: {name}"
            assert len(pd.read_csv(evaluation / "prototype/test_predictions.csv")) == len(frames["test"])
        smoke_rows.append(metrics)
        print(f"PASS: {model_name}: one epoch, checkpoint evaluation and expected artifacts verified.")
    smoke_summary = pd.concat(smoke_rows, ignore_index=True)
    smoke_summary.to_csv(SMOKE_OUTPUT / "smoke_comparison.csv", index=False)
    display(smoke_summary)
    print("SMOKE TEST PASSED. Results:", SMOKE_OUTPUT)
    print("These one-epoch results check execution; they are not the final experiment results.")
else:
    print('Smoke test skipped. Set MODE="smoke" and rerun setup to enable it.')


Launch preflight passed: 33126 images and leakage audit.
Launch preflight passed: 28126 images and leakage audit.
Running: ['/usr/bin/python3', '-u', '/content/PatternAnalysis-2026-5ce0a02/recognition/Siamese_Nandos/train.py', '--model', 'baseline', '--splits', '/content/PatternAnalysis-2026-5ce0a02/recognition/Siamese_Nandos/splits', '--images', '/content/isic2020/jpeg', '--workers', '2', '--device', 'cuda', '--output', '/content/drive/MyDrive/COMP3710/isic_runs/20261117T100942Z_87ba2d49/smoke_2c60c256/baseline/training', '--epochs', '1', '--log-every', '25'] 
Log: /content/drive/MyDrive/COMP3710/isic_runs/20261117T100942Z_87ba2d49/smoke_baseline_train_102015_3dbfd067.log
Starting baseline training with train_baseline.py
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth

  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 50%|█████     | 22.4M/44.7M [00:00<00:00, 234MB/s]
100%|██████████| 44.7M/44.7M [00:00<00

,model,threshold_policy,threshold,validation_recall_at_threshold,validation_precision_at_threshold,validation_specificity_at_threshold,test_images,test_melanomas,test_accuracy,test_macro_f1,...,test_referral_rate,test_automatic_coverage,test_automatic_images,test_automatic_accuracy,test_unreferred_melanoma_misses,test_referred_melanomas,test_calibration_ece_10_bins,checkpoint_epoch,checkpoint_validation_ap,device
0,Baseline ResNet-18,val_specificity80_max_sensitivity,0.493889,0.663043,NaN,0.813454,5000,89,0.8122,0.505791,...,0.1914,0.8086,4043,0.874351,14,26,0.252361,1,0.181860,NVIDIA L4
1,Siamese + auxiliary head,val_specificity80_max_sensitivity,0.003915,0.826087,NaN,0.824804,5000,89,0.8016,0.508178,...,0.2048,0.7952,3976,0.794769,2,19,0.042519,0,0.193432,NVIDIA L4


SMOKE TEST PASSED. Results: /content/drive/MyDrive/COMP3710/isic_runs/20261117T100942Z_87ba2d49/smoke_2c60c256
These one-epoch results check execution; they are not the final experiment results.


## Full experiment (skipped in smoke mode)


In [11]:
if MODE == "train":
    for model_name in MODELS:
        train_output = OUTPUT / "training" / model_name
        run_script("train.py", ["--model", model_name, "--splits", SPLITS,
                   "--images", IMAGE_DIR, 
                   "--output", train_output,
                   "--cache-images",  # cache resized images in CPU RAM
                   "--workers", WORKERS,
                   "--device", "cuda",
                   "--log-every", LOG_EVERY,
                   *(["--resume"] if RESUME_RUN is not None else [])], f"{model_name}_train.log")
        CHECKPOINTS[model_name] = train_output / "best.pt"
elif MODE == "evaluate":
    print("Using supplied checkpoints; no retraining.")


Launch preflight passed: 28126 images and leakage audit.
Running: ['/usr/bin/python3', '-u', '/content/PatternAnalysis-2026-5ce0a02/recognition/Siamese_Nandos/train.py', '--model', 'baseline', '--splits', '/content/PatternAnalysis-2026-5ce0a02/recognition/Siamese_Nandos/splits', '--images', '/content/isic2020/jpeg', '--output', '/content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4/training/baseline', '--cache-images', '--workers', '2', '--device', 'cuda', '--log-every', '25'] 
Log: /content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4/baseline_train_132954_73dc200e.log
Starting baseline training with train_baseline.py
Cached 28,126 train/validation images (3.94 GB in RGB uint8 pixels).
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth

  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 17%|█▋        | 7.62M/44.7M [00:00<00:00, 79.8MB/s]
 67%|██████▋   | 30.1M/44.7M [00:00<00:00, 17

In [12]:
if MODE != "smoke":
    checkpoint_hashes = {}
    for model_name in MODELS:
        checkpoint = CHECKPOINTS[model_name]
        checkpoint_hashes[model_name] = {"path": str(checkpoint), "sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest()}
        run_script("predict.py", [
            "--model", model_name,
            "--checkpoint", checkpoint,
            "--splits", SPLITS,
            "--images", IMAGE_DIR,
            "--output", OUTPUT / f"{model_name}_{RUN_ID}",
            "--batch-size", BATCH_SIZE,
            "--workers", WORKERS,
            "--device", "cuda",
            "--threshold-policy", "val_specificity80_max_sensitivity",
            *(["--prototype"] if model_name == "siamese_aux" else []),
        ], f"{model_name}_evaluation.log")
    (OUTPUT / f"checkpoints_{RUN_ID}.json").write_text(json.dumps(checkpoint_hashes, indent=2))
    summary = pd.concat([pd.read_csv(OUTPUT / f"{name}_{RUN_ID}" / "test_metrics.csv") for name in MODELS], ignore_index=True)
    summary.to_csv(OUTPUT / f"comparison_{RUN_ID}.csv", index=False)
    display(summary)
    print("Experiment completed. Results:", OUTPUT)


Launch preflight passed: 9938 images and leakage audit.
Running: ['/usr/bin/python3', '-u', '/content/PatternAnalysis-2026-5ce0a02/recognition/Siamese_Nandos/predict.py', '--model', 'baseline', '--checkpoint', '/content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4/training/baseline/best.pt', '--splits', '/content/PatternAnalysis-2026-5ce0a02/recognition/Siamese_Nandos/splits', '--images', '/content/isic2020/jpeg', '--output', '/content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4/baseline_20261117T132323Z_613158f4', '--batch-size', '64', '--workers', '2', '--device', 'cuda', '--threshold-policy', 'val_specificity80_max_sensitivity'] 
Log: /content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4/baseline_evaluation_160410_a0d37d08.log
Model: Baseline ResNet-18
Device: NVIDIA A100-SXM4-40GB
Checkpoint: /content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4/training/baseline/best.pt (epoch 9)
Validation threshold policy: val_specificity8

,model,threshold_policy,threshold,validation_recall_at_threshold,validation_precision_at_threshold,validation_specificity_at_threshold,test_images,test_melanomas,test_accuracy,test_macro_f1,...,test_referral_rate,test_automatic_coverage,test_automatic_images,test_automatic_accuracy,test_unreferred_melanoma_misses,test_referred_melanomas,test_calibration_ece_10_bins,checkpoint_epoch,checkpoint_validation_ap,device
0,Baseline ResNet-18,val_specificity80_max_sensitivity,0.187728,0.815217,NaN,0.822534,5000,89,0.8084,0.511503,...,0.1980,0.8020,4010,0.833915,11,15,0.100362,9,0.215573,NVIDIA A100-SXM4-40GB
1,Siamese + auxiliary head,val_specificity80_max_sensitivity,0.000116,0.739130,NaN,0.804787,5000,89,0.8084,0.505995,...,0.2024,0.7976,3988,0.798144,7,17,0.025581,10,0.211739,NVIDIA A100-SXM4-40GB


Experiment completed. Results: /content/drive/MyDrive/COMP3710/isic_runs/20261117T132323Z_613158f4
